# Language ID Evaluation Report

It loads the generated CSV and JSON outputs in `reports/analysis/` and presents the deployed
baseline classifier results as quick visual tables.


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if (ROOT / "reports").exists():
    PROJECT_ROOT = ROOT
else:
    PROJECT_ROOT = ROOT.parent.parent

ANALYSIS_DIR = PROJECT_ROOT / "reports" / "analysis"
ANALYSIS_DIR


PosixPath('/home/joels/PycharmProjects/nlp-adaptive-tutor/reports/analysis')

In [2]:
summary = json.loads((ANALYSIS_DIR / "language_id_summary.json").read_text())
confusion = pd.read_csv(ANALYSIS_DIR / "language_id_confusion.csv", index_col="true_label")
metrics = pd.read_csv(ANALYSIS_DIR / "language_id_report.csv")

summary_df = pd.DataFrame(
    [
        {"metric": "samples", "value": summary["n_samples"]},
        {"metric": "accuracy", "value": round(summary["accuracy"], 4)},
        {"metric": "macro_f1", "value": round(summary["macro_f1"], 4)},
    ]
)

summary_df


,metric,value
0,samples,1500.000
1,accuracy,0.986
2,macro_f1,0.986


In [3]:
def bar_cell(value, max_value, color):
    value = float(value)
    pct = 0.0 if max_value == 0 else (100.0 * value / max_value)
    return (
        "<div style='display:flex;align-items:center;gap:8px'>"
        "<div style='width:140px;height:12px;background:#edf2f7;border-radius:999px;overflow:hidden'>"
        f"<div style='width:{pct:.1f}%;height:12px;background:{color}'></div>"
        "</div>"
        f"<span style='font-family:monospace'>{value:.3f}</span>"
        "</div>"
    )


def render_bar_table(df, value_cols, extra_cols=None, colors=None):
    extra_cols = extra_cols or []
    colors = colors or {}
    max_values = {col: max(float(df[col].max()), 1e-9) for col in value_cols}
    headers = extra_cols + value_cols
    html = ["<table style='border-collapse:collapse;width:100%;font-size:14px'>"]
    html.append("<thead><tr>")
    for col in headers:
        html.append(
            f"<th style='text-align:left;border-bottom:2px solid #cbd5e1;padding:8px'>{col}</th>"
        )
    html.append("</tr></thead><tbody>")
    for _, row in df.iterrows():
        html.append("<tr>")
        for col in extra_cols:
            html.append(
                f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;vertical-align:top'>{row[col]}</td>"
            )
        for col in value_cols:
            html.append(
                "<td style='padding:8px;border-bottom:1px solid #e2e8f0'>"
                + bar_cell(row[col], max_values[col], colors.get(col, "#2563eb"))
                + "</td>"
            )
        html.append("</tr>")
    html.append("</tbody></table>")
    display(HTML("".join(html)))


def render_confusion_table(df):
    max_value = max(int(df.to_numpy().max()), 1)
    html = ["<table style='border-collapse:collapse;font-size:14px'>"]
    html.append("<thead><tr><th style='padding:8px'></th>")
    for col in df.columns:
        html.append(
            f"<th style='padding:8px;border-bottom:2px solid #cbd5e1'>{col}</th>"
        )
    html.append("</tr></thead><tbody>")
    for idx, row in df.iterrows():
        html.append("<tr>")
        html.append(
            f"<th style='padding:8px;text-align:left;border-right:2px solid #cbd5e1'>{idx}</th>"
        )
        for value in row:
            alpha = 0.12 + (float(value) / max_value) * 0.78
            html.append(
                "<td style='padding:10px;text-align:center;color:#0f172a;"
                f"background:rgba(37,99,235,{alpha:.2f});border:1px solid #ffffff'>{int(value)}</td>"
            )
        html.append("</tr>")
    html.append("</tbody></table>")
    display(HTML("".join(html)))


## Confusion Matrix

The heat-style table below is useful for screenshots because the darker cells make the diagonal
dominance easy to see immediately.


In [4]:
render_confusion_table(confusion)


,English,Polish,Spanish
English,497,2,1
Polish,5,493,2
Spanish,6,5,489


## Per-Language Precision, Recall, and F1

These bars make it easy to compare the three deployed languages at a glance.


In [5]:
render_bar_table(
    metrics,
    value_cols=["precision", "recall", "f1_score"],
    extra_cols=["label", "support"],
    colors={
        "precision": "#2563eb",
        "recall": "#059669",
        "f1_score": "#dc2626",
    },
)


label,support,precision,recall,f1_score
English,500,0.978,0.994,0.986
Polish,500,0.986,0.986,0.986
Spanish,500,0.994,0.978,0.986


In [6]:
metrics.sort_values("recall", ascending=False)


,label,precision,recall,f1_score,support
0,English,0.978346,0.994,0.986111,500
1,Polish,0.986000,0.986,0.986000,500
2,Spanish,0.993902,0.978,0.985887,500
